# Clase 3 — Visualización y Análisis Controlado (Casos Sintéticos)
### Análisis de Datos en Python · Curso intensivo

**Idea central:** hoy no aprendemos a "hacer gráficos". Aprendemos a **usar la vista como
instrumento de verificación** de la estadística de la Clase 1. Cada gráfico es una *hipótesis*
que después confirmamos o refutamos con un número.

Trabajamos con **datos sintéticos**: conocemos la verdad (la media real, la σ real, la
correlación real), así que podemos comprobar si nuestros métodos la recuperan. Ese lujo
desaparece en la Clase 4 (datos reales).

**Datasets de hoy** (generados con semilla fija, verdad conocida):
| Archivo | Fenómeno diseñado | Concepto de la Clase 1 que reaparece |
|---|---|---|
| `dataset_A_asimetria.csv` | Asimetría fuerte a la derecha | Media vs. mediana, imputación |
| `dataset_B_outliers.csv` | Outliers de alto apalancamiento | Dispersión, IQR, correlación |
| `dataset_C_confusion.csv` | Correlación espuria por confusión | Correlación, causalidad |


## 0 · Preparación del entorno

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)

# TODO: carga los tres CSV en df_A, df_B, df_C con pd.read_csv(...)
df_A = ...
df_B = ...
df_C = ...
# TODO: imprime la forma (shape) de cada uno para comprobar que cargaron

## Bloque 0 (10 min) · ¿Por qué visualizar? El cuarteto de Anscombe

**El "por qué".** Cuatro conjuntos con **la misma media, la misma desviación y la misma
correlación de Pearson**... y cuatro realidades completamente distintas. Moraleja que guía
toda la clase: *un resumen numérico nunca sustituye a mirar los datos.*

In [ ]:
# Cuarteto de Anscombe (valores clásicos, ya provistos)
x123 = np.array([10,8,13,9,11,14,6,4,12,7,5], float)
x4   = np.array([8,8,8,8,8,8,8,19,8,8,8], float)
ans = {
    "I":   (x123, np.array([8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68])),
    "II":  (x123, np.array([9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74])),
    "III": (x123, np.array([7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73])),
    "IV":  (x4,   np.array([6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89])),
}
# TODO: para cada set imprime media_x, media_y, std_y y r de Pearson.
# ¿Son casi idénticos los cuatro resúmenes?

In [ ]:
# TODO: dibuja un scatter de cada uno de los 4 sets en una rejilla 2x2
#       y superpón la recta de ajuste (usa np.polyfit(x, y, 1)).
#       Observa: ¿la misma recta describe bien los cuatro casos?

> **Conclusión del bloque.** Nunca confíes en `r` (ni en la media) sin mirar la nube.
> El set II es curvo, el III está dominado por un outlier, el IV es una recta vertical + 1 punto.

## Bloque 1 (20 min) · Histogramas → Dataset A (asimetría)

**El "por qué".** El histograma es la versión gráfica de la **tabla de frecuencias** de la
Clase 1. En una distribución asimétrica a la derecha, la **media se desplaza hacia la cola**
y deja de representar al hogar típico; la **mediana** se mantiene en el centro de masa de los
datos. Por eso, para *imputar* y para *reportar*, la mediana suele ser la elección honesta.

### 🔁 Ejercicio repetitivo #1
Recalcula media y mediana (Clase 1), márcalas sobre el histograma y decide con qué imputar.

In [ ]:
col = df_A["ingreso_mensual"]
# TODO: calcula media y mediana; imprime cuántos nulos hay.
# TODO: dibuja el histograma (prueba bins=40) y traza dos líneas verticales
#       (plt.axvline) para la media y la mediana.
# Pregunta: ¿cuál queda más a la derecha y por qué?

In [ ]:
# TODO: crea dos versiones imputando los nulos con fillna():
#   imp_media   = col.fillna( ??? )
#   imp_mediana = col.fillna( ??? )
# Compara la media y la mediana resultantes. ¿Cuál deforma menos la distribución?

> **Decisión justificada:** en esta columna asimétrica imputamos con **mediana**. Rellenar
> con la media empuja los valores faltantes hacia arriba (recuerda: los nulos eran más
> frecuentes en ingresos altos), amplificando el sesgo.

## Bloque 2 (20 min) · Boxplot y atípicos → Dataset B

**El "por qué".** El boxplot codifica los **cuartiles y el IQR** de la Clase 1 y define
formalmente un atípico: fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`. Pero detectar no es borrar:
un atípico puede ser un error de captura **o** el dato más informativo. Aquí son errores de
digitación (se anotó 40 h en vez de 4) y **degradan la correlación**.

### 🔁 Ejercicio repetitivo #2
Detecta outliers con la regla IQR, decide qué hacer y **recalcula la correlación** (Clase 1).

In [ ]:
def limites_iqr(s):
    # TODO: devuelve (Q1 - 1.5*IQR, Q3 + 1.5*IQR) usando s.quantile(...)
    ...

# TODO: dibuja el boxplot de 'horas_estudio' y 'puntuacion'.
# TODO: usa limites_iqr() para marcar las filas atípicas de 'horas_estudio'
#       e imprímelas. ¿Cuántas hay? ¿Qué tienen de raro?

In [ ]:
# TODO: calcula la correlación de Pearson horas_estudio vs puntuacion
#       (a) con los outliers y (b) tras quitarlos con df_B.drop(outliers.index).
# TODO: compara también la desviación estándar antes y después.
# Pregunta: ¿cuánto cambió la conclusión por 6 puntos mal capturados?

> **Decisión justificada:** documentamos y removemos (o corregimos) esos 6 registros porque
> son errores de captura verificables, no observaciones legítimas. La correlación pasa de
> engañosa a real. Siempre se justifica *por qué* se toca un dato.

## Bloque 3 (20 min) · Scatter y correlación → Anscombe revisitado

**El "por qué".** Un mismo `r` puede esconder una recta, una curva o un outlier dominante.
Por eso el scatter **precede** al coeficiente, nunca al revés. Pearson mide relación *lineal*;
Spearman capta monotonía (útil cuando la relación es curva pero creciente).

### 🔁 Ejercicio repetitivo #3
Confirma que los 4 sets de Anscombe tienen (casi) el mismo `r` pese a nubes distintas.

In [ ]:
# TODO: recorre los 4 sets de 'ans' y calcula para cada uno
#       la correlación de Pearson y la de Spearman (Pearson sobre los rangos, .rank()).
# ¿Confirmas que la Pearson es casi idéntica en los cuatro?

## Bloque integrador (30 min) · Ciclo iterativo sobre los 3 casos

**El "por qué".** El análisis real es **iterativo**: *conjeturar → graficar → medir → corregir
→ volver a medir*. Aquí el foco es el Dataset C, donde una correlación fuerte **se desvanece**
al condicionar por una tercera variable (confusión / semilla de la paradoja de Simpson).

**Historia:** en una ciudad costera, los días de más **ventas de helados** coinciden con más
**ahogamientos**. ¿Comer helado es peligroso? Conjetura, grafica y decide.

In [ ]:
# Paso 1 — TODO: mide corr(ventas_helados, ahogamientos) y dibuja el scatter.
# ¿La correlación sugiere causalidad?

In [ ]:
# Paso 2 — TODO: repite el scatter pero colorea los puntos por 'temperatura_C'
#          (argumento c=..., cmap='coolwarm') y añade plt.colorbar().
# ¿Qué patrón aparece con el color?

In [ ]:
# Paso 3 — TODO: crea franjas de temperatura con pd.cut(..., bins=[0,15,25,35])
#          y calcula la correlación helados-ahogamientos DENTRO de cada franja
#          (groupby + apply). ¿Se mantiene la correlación o desaparece?

> **Cierre del ciclo.** La temperatura es la **causa común**: días calurosos → más helados
> y más gente en el agua. Correlación ≠ causalidad. Reúso simultáneo de: correlación,
> agrupamiento, y pensamiento crítico. *Repite este ciclo con A (¿imputar cambia la media?)
> y con B (¿los outliers cambian la pendiente?).*

## Bloque de síntesis (20 min) · Elegir el gráfico correcto y no mentir

**El "por qué".** El tipo de variable determina el gráfico: **1 variable numérica → histograma
/ boxplot**; **2 numéricas → scatter**; **categórica vs numérica → boxplot por grupo**;
**categórica → barras**. Y un gráfico es un *argumento*: los ejes truncados exageran
diferencias. Elegir mal el gráfico es un error analítico, no estético.

In [ ]:
# TODO: reproduce dos barras con valores [102, 100].
#   Gráfico 1: set_ylim(99, 103)  -> exagera la diferencia
#   Gráfico 2: set_ylim(0, 110)   -> muestra la diferencia real
# Comenta qué ves.

### ✅ Checklist de visualización (para llevar a la Clase 4)
1. ¿Qué **tipo** de variable(s) tengo? → elige el gráfico adecuado.
2. ¿El **histograma** confirma la forma que asumí (simetría, modas)?
3. ¿Hay **atípicos** (boxplot/IQR)? ¿Son error o señal? Justifica la decisión.
4. Antes de confiar en `r`, **¿grafiqué la nube?** (recuerda Anscombe).
5. ¿Hay una **variable oculta** que explique la correlación? (confusión).
6. ¿Mis **ejes** cuentan la verdad o exageran?

**Puente a la Clase 4:** estas tres alertas — asimetría, atípicos y confusión — reaparecerán
en datos reales, donde ya no conoceremos la verdad de antemano.